# КМ2 — Распознавание персональных данных

In [1]:
import importlib.metadata
import importlib.util
import subprocess
import sys

packages = [
    ("torch", "torch>=2.6", None),
    ("transformers", "transformers==4.51.3", "4.51.3"),
    ("sklearn", "scikit-learn==1.6.1", "1.6.1"),
    ("pymorphy3", "pymorphy3==2.0.6", "2.0.6"),
    ("faker", "Faker==37.1.0", "37.1.0"),
    ("requests", "requests==2.32.3", "2.32.3"),
]
missing = []
for module, requirement, expected_version in packages:
    distribution = requirement.split("==")[0].split(">=")[0]
    if importlib.util.find_spec(module) is None:
        missing.append(requirement)
    elif expected_version and importlib.metadata.version(distribution) != expected_version:
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])


In [2]:
import json
import os
import zipfile
from pathlib import Path

WORKDIR = Path.cwd()
if not (WORKDIR / "data").exists():
    WORKDIR = WORKDIR / "km2_variant12"
    WORKDIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = WORKDIR / "models" / "person"
DATA_DIR = WORKDIR / "data"
RESULTS_DIR = WORKDIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Каталог:", WORKDIR)


Каталог: C:\Users\artizi\Desktop\kp_tzi


In [3]:
import json

import math

import re

from dataclasses import dataclass

from pathlib import Path

LABELS = ('SURNAME',
 'NAME',
 'INN',
 'PASSPORT',
 'BANK_CARD',
 'OGRNIP',
 'IP',
 'EMAIL',
 'BIRTH_CERT',
 'KLADR')

RUSSIAN = dict(zip(LABELS, ('Фамилия', 'Имя', 'ИНН', 'Паспорт', 'Банковская карта', 'ОГРНИП', 'IP-адрес', 'Электронная почта', 'Свидетельство о рождении', 'Код КЛАДР')))

TOKEN_RE = re.compile('[^\\W\\d_]+(?:[-‐‑][^\\W\\d_]+)*|\\d+|[^\\w\\s]', re.UNICODE)

@dataclass(frozen=True)
class Span:
    label: str
    start: int
    end: int
    score: float = 1.0
    source: str = 'rule'

    def triple(self):
        return [self.label, self.start, self.end]

def tokenize(text):
    if not isinstance(text, str):
        raise TypeError('text must be str')
    return [(m.group(), m.start(), m.end()) for m in TOKEN_RE.finditer(text)]

def validate_spans(text, spans):
    if not isinstance(text, str):
        raise TypeError('text must be str')
    result = []
    for item in spans:
        s = item if isinstance(item, Span) else Span(*item)
        if s.label not in LABELS:
            raise ValueError(f'Unknown label: {s.label}')
        if type(s.start) is not int or type(s.end) is not int or (not 0 <= s.start < s.end <= len(text)):
            raise ValueError('Invalid source coordinates')
        result.append(s)
    result.sort(key=lambda s: (s.start, s.end))
    if any((a.end > b.start for a, b in zip(result, result[1:]))):
        raise ValueError('Overlapping source intervals')
    return result

def to_bio(text, spans):
    tokens = tokenize(text)
    tags = ['O'] * len(tokens)
    for s in validate_spans(text, spans):
        ids = [i for i, (_, a, b) in enumerate(tokens) if a < s.end and b > s.start]
        if not ids or tokens[ids[0]][1] != s.start or tokens[ids[-1]][2] != s.end:
            raise ValueError(f'Span is not aligned with evaluation tokens: {s}')
        for j, i in enumerate(ids):
            tags[i] = ('B-' if j == 0 else 'I-') + s.label
    return (tokens, tags)

def from_bio(tokens, tags, source='model'):
    result = []
    active = None
    for (_, a, b), tag in zip(tokens, tags):
        if tag == 'O':
            active = None
            continue
        prefix, label = tag.split('-', 1)
        if prefix == 'I' and active is not None and (result[active].label == label):
            old = result[active]
            result[active] = Span(label, old.start, b, source=source)
        else:
            result.append(Span(label, a, b, source=source))
            active = len(result) - 1
    return result

def resolve(candidates):
    priority = {'EMAIL': 100, 'BIRTH_CERT': 95, 'IP': 90, 'KLADR': 85, 'OGRNIP': 80, 'INN': 75, 'BANK_CARD': 70, 'PASSPORT': 65}
    accepted = []
    for s in sorted(set(candidates), key=lambda s: (-priority.get(s.label, 0), -s.score, -(s.end - s.start), s.start)):
        if all((s.end <= x.start or x.end <= s.start for x in accepted)):
            accepted.append(s)
    return sorted(accepted, key=lambda s: s.start)

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    def finite(x):
        if isinstance(x, float) and (not math.isfinite(x)):
            return None
        if isinstance(x, dict):
            return {k: finite(v) for k, v in x.items()}
        if isinstance(x, (list, tuple)):
            return [finite(v) for v in x]
        return x
    path.write_text(json.dumps(finite(value), ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def write_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


In [4]:
'Deterministic candidates; checksum validity is not registry existence.'

import ipaddress

import re

def digits(s):
    return ''.join((c for c in s if c in '0123456789'))

def inn_complete(base):

    def check(v, w):
        return str(sum((int(a) * b for a, b in zip(v, w))) % 11 % 10)
    if len(base) == 9:
        return base + check(base, (2, 4, 10, 3, 5, 9, 4, 6, 8))
    if len(base) == 10:
        first = base + check(base, (7, 2, 4, 10, 3, 5, 9, 4, 6, 8))
        return first + check(first, (3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8))
    raise ValueError('INN base must have 9 or 10 digits')

def valid_inn(s):
    s = digits(s)
    return len(s) in (10, 12) and len(set(s)) > 1 and (inn_complete(s[:-1] if len(s) == 10 else s[:-2]) == s)

def valid_ogrnip(s):
    return len(s) == 15 and s.isascii() and s.isdigit() and (s[0] in '34') and (int(s[3:5]) in range(1, 100)) and (int(s[-1]) == int(s[:-1]) % 13 % 10)

def luhn_sum(s):
    return sum(((2 * int(c) - 9 if 2 * int(c) > 9 else 2 * int(c)) if i % 2 else int(c) for i, c in enumerate(s[::-1])))

def valid_card(s):
    s = digits(s)
    return 13 <= len(s) <= 19 and len(set(s)) > 1 and (luhn_sum(s) % 10 == 0)

def complete_card(base):
    return base + str(-luhn_sum(base + '0') % 10)

EMAIL = re.compile('(?<![\\w.+-])[\\w]+(?:[.+-][\\w]+)*@(?:[\\w](?:[\\w-]*[\\w])?\\.)+[a-zа-яё]{2,63}(?![\\w-])', re.I)

IPV4 = re.compile('(?<![\\w.])(?:\\d{1,3}\\.){3}\\d{1,3}(?!\\w|\\.\\d)')

IPV6 = re.compile('(?<![\\w:])(?:[0-9a-f]{0,4}:){2,7}[0-9a-f]{0,4}(?![\\w:])', re.I)

INN = re.compile('(?<!\\d)(?:\\d{12}|\\d{10})(?!\\d)')

OGRNIP = re.compile('(?<!\\d)[34]\\d{14}(?!\\d)')

CARD = re.compile('(?<!\\d)\\d(?:[ -]?\\d){12,18}(?![ -]?\\d)')

PASSPORT = re.compile('(?<!\\d)(?:\\d{2}[ ]?\\d{2})[ ]\\d{6}(?!\\d)')

BIRTH = re.compile('(?<![\\w])[IVXLCDMІ]{1,6}[-–][А-ЯЁ]{2}\\s*(?:№\\s*)?\\d{6}(?!\\d)')

KLADR = re.compile('(?<!\\d)(?:\\d{19}|\\d{17}|\\d{13})(?!\\d)')

def structured(text):
    out = []

    def add(label, match):
        out.append(Span(label, match.start(), match.end()))
    for m in EMAIL.finditer(text):
        add('EMAIL', m)
    for pattern in (IPV4, IPV6):
        for m in pattern.finditer(text):
            try:
                ipaddress.ip_address(m.group())
            except ValueError:
                continue
            add('IP', m)
    for m in BIRTH.finditer(text):
        context = text[max(0, m.start() - 80):m.start()].lower()
        if 'брак' not in context or 'рожд' in context:
            add('BIRTH_CERT', m)
    for m in KLADR.finditer(text):
        if re.search('кладр\\s*[:=№-]?\\s*$', text[max(0, m.start() - 35):m.start()], re.I) and 1 <= int(m.group()[:2]) <= 99:
            add('KLADR', m)
    for m in OGRNIP.finditer(text):
        if valid_ogrnip(m.group()):
            add('OGRNIP', m)
    for m in INN.finditer(text):
        if valid_inn(m.group()):
            add('INN', m)
    for m in CARD.finditer(text):
        if re.search('(?:огрнип|инн|кладр)\\s*[:=№]?\\s*$', text[max(0, m.start() - 30):m.start()], re.I):
            continue
        if valid_card(m.group()):
            add('BANK_CARD', m)
    for m in PASSPORT.finditer(text):
        if 1 <= int(digits(m.group())[:2]) <= 99:
            add('PASSPORT', m)
    for m in re.finditer('(?<!\\d)\\d{10}(?!\\d)', text):
        if re.search('паспорт(?:а|ом)?\\s*[:№]?\\s*$', text[max(0, m.start() - 30):m.start()], re.I) and 1 <= int(m.group()[:2]) <= 99:
            add('PASSPORT', m)
    return resolve(out)


In [5]:
from pathlib import Path

class PersonModel:
    """Word-level decisions from first-subtoken logits, overlapping windows."""

    def __init__(self, model_dir='models/person', device=None, batch_size=8):
        import torch
        from transformers import AutoModelForTokenClassification, AutoTokenizer
        self.torch = torch
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = AutoTokenizer.from_pretrained(str(model_dir), local_files_only=True)
        self.model = AutoModelForTokenClassification.from_pretrained(str(model_dir), local_files_only=True).to(self.device).eval()
        self.batch_size = batch_size

    def predict_many(self, texts):
        torch = self.torch
        token_lists = [tokenize(t) for t in texts]
        result = []
        for tokens in token_lists:
            if not tokens:
                result.append([])
                continue
            enc = self.tokenizer([t[0] for t in tokens], is_split_into_words=True, truncation=True, max_length=256, stride=48, return_overflowing_tokens=True, padding=True, return_tensors='pt')
            best = {}
            for start in range(0, len(enc['input_ids']), self.batch_size):
                batch = {k: v[start:start + self.batch_size].to(self.device) for k, v in enc.items() if k in ('input_ids', 'attention_mask', 'token_type_ids')}
                with torch.inference_mode():
                    logits = self.model(**batch).logits.float().cpu()
                for k, scores in enumerate(logits):
                    ids = enc.word_ids(start + k)
                    for i, wid in enumerate(ids):
                        if wid is None or (i > 0 and ids[i - 1] == wid):
                            continue
                        centrality = min(i, int(enc['attention_mask'][start + k].sum()) - 1 - i)
                        if wid not in best or centrality > best[wid][0]:
                            best[wid] = (centrality, self.model.config.id2label[int(scores[i].argmax())])
            tags = [best.get(i, (0, 'O'))[1] for i in range(len(tokens))]
            result.append(from_bio(tokens, tags))
        return result

    def __call__(self, text):
        return self.predict_many([text])[0]

class HybridDetector:

    def __init__(self, model_dir='models/person', device=None, rules_only=False):
        self.person = None if rules_only else PersonModel(model_dir, device)

    def predict_many(self, texts):
        person = self.person.predict_many(texts) if self.person else [[] for _ in texts]
        return [resolve(structured(text) + p) for text, p in zip(texts, person)]

    def __call__(self, text):
        if not isinstance(text, str):
            raise TypeError('text must be str')
        return self.predict_many([text])[0]

    def recognize(self, text):
        return [s.triple() for s in self(text)]


In [6]:
import time

from collections import Counter

from sklearn.metrics import classification_report, confusion_matrix

def counts_metrics(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {'tp': tp, 'fp': fp, 'fn': fn, 'support': tp + fn, 'precision': p, 'recall': r, 'f1': 2 * p * r / (p + r) if p + r else 0.0}

def evaluate(rows, predictions):
    if len(rows) != len(predictions):
        raise ValueError('Mismatched prediction count')
    gold_tags = []
    pred_tags = []
    gold_attr = []
    pred_attr = []
    errors = []
    entity_counts = {l: Counter() for l in LABELS}
    for row, pred in zip(rows, predictions):
        _, g = to_bio(row['text'], row['entities'])
        _, p = to_bio(row['text'], pred)
        gold_tags.extend(g)
        pred_tags.extend(p)
        gold_attr.extend((t.split('-', 1)[1] if t != 'O' else 'O' for t in g))
        pred_attr.extend((t.split('-', 1)[1] if t != 'O' else 'O' for t in p))
        gs = {tuple(x) for x in row['entities']}
        ps = {tuple(x.triple()) for x in pred}
        for label in LABELS:
            gl = {s for s in gs if s[0] == label}
            pl = {s for s in ps if s[0] == label}
            entity_counts[label].update(tp=len(gl & pl), fp=len(pl - gl), fn=len(gl - pl))
        if gs != ps:
            errors.append({'id': row['id'], 'source': row['source'], 'text': row['text'], 'false_positive': [list(s) for s in sorted(ps - gs)], 'false_negative': [list(s) for s in sorted(gs - ps)]})
    bio_labels = [p + l for l in LABELS for p in ('B-', 'I-') if p + l in set(gold_tags)]
    predicted_only = sorted(set(pred_tags) - set(gold_tags) - {'O'})
    bio_labels += predicted_only
    token = classification_report(gold_tags, pred_tags, labels=bio_labels, output_dict=True, zero_division=0)
    attribute = classification_report(gold_attr, pred_attr, labels=list(LABELS), output_dict=True, zero_division=0)
    exact = {l: counts_metrics(c['tp'], c['fp'], c['fn']) for l, c in entity_counts.items()}
    overall = counts_metrics(sum((x['tp'] for x in exact.values())), sum((x['fp'] for x in exact.values())), sum((x['fn'] for x in exact.values())))
    per_label = {}
    for l in LABELS:
        tp = sum((g == l and p == l for g, p in zip(gold_attr, pred_attr)))
        fp = sum((g != l and p == l for g, p in zip(gold_attr, pred_attr)))
        fn = sum((g == l and p != l for g, p in zip(gold_attr, pred_attr)))
        per_label[l] = counts_metrics(tp, fp, fn)
    metrics = {'documents': len(rows), 'tokens': len(gold_tags), 'main_metric': 'BIO token classification, O excluded from averaging; O positions retained', 'bio': token, 'attribute_tokens': attribute, 'per_attribute': per_label, 'exact_entities': exact, 'exact_micro': overall, 'documents_with_errors': len(errors), 'thresholds': {'weighted_required': 0.85, 'macro_required': 0.8, 'passed': token['weighted avg']['f1-score'] >= 0.85 and token['macro avg']['f1-score'] >= 0.8}, 'confusion_labels': ['O'] + list(LABELS), 'confusion_matrix': confusion_matrix(gold_attr, pred_attr, labels=['O'] + list(LABELS)).tolist()}
    return (metrics, errors)


In [7]:
from functools import lru_cache
@lru_cache(maxsize=1)
def morph():
    import pymorphy3
    return pymorphy3.MorphAnalyzer()


In [8]:
'Independent gold annotation by source spans / insertion offsets, never detector.'

import hashlib

import random

import re

import sys

import urllib.request

import zipfile

from collections import Counter

from pathlib import Path

from functools import lru_cache

@lru_cache(maxsize=1)
def morph():
    import pymorphy3
    return pymorphy3.MorphAnalyzer()

ROOT = WORKDIR

def fact_documents(folder):
    rows = []
    conflicts = []
    for f in sorted(folder.glob('book_*.txt')):
        text = f.read_text(encoding='utf-8-sig')
        persons = set()
        for line in f.with_suffix('.objects').read_text(encoding='utf-8-sig').splitlines():
            fields = line.split('#', 1)[0].split()
            if len(fields) > 2 and fields[1] == 'Person':
                persons.update(fields[2:])
        spans = []
        for line in f.with_suffix('.spans').read_text(encoding='utf-8-sig').splitlines():
            fields = line.split('#', 1)[0].split()
            if len(fields) < 4 or fields[0] not in persons or fields[1] not in ('name', 'surname'):
                continue
            start, length = map(int, fields[2:4])
            spans.append(Span('NAME' if fields[1] == 'name' else 'SURNAME', start, start + length, source='gold'))
        spans = sorted(set(spans), key=lambda s: (s.start, s.end, s.label))
        selected = resolve(spans)
        if len(spans) != len(selected):
            conflicts.append({'id': f.stem, 'before': len(spans), 'after': len(selected)})
        toks = tokenize(text)
        aligned = []
        for s in selected:
            ts = [(a, b) for _, a, b in toks if a < s.end and b > s.start]
            if ts:
                aligned.append(Span(s.label, ts[0][0], ts[-1][1], source='gold'))
        aligned = resolve(aligned)
        to_bio(text, aligned)
        rows.append({'id': f'fact_{folder.name}_{f.stem}', 'source': 'factRuEval2016', 'text': text, 'entities': [s.triple() for s in aligned]})
    return (rows, conflicts)

def render(template, values):
    text = ''
    spans = []
    for part in re.split('(\\{[A-Z_]+\\})', template):
        if part.startswith('{') and part.endswith('}'):
            label = part[1:-1]
            v = values[label]
            start = len(text)
            text += v
            spans.append([label, start, len(text)])
        else:
            text += part
    return (text, spans)

TEMPLATES = {'train': ['{NAME} {SURNAME} подал заявление. ИНН {INN}; паспорт {PASSPORT}; карта '
           '{BANK_CARD}. ОГРНИП {OGRNIP}, IP {IP}, почта {EMAIL}. Свидетельство о '
           'рождении {BIRTH_CERT}. Код КЛАДР {KLADR}.',
           'Мы получили письмо от {NAME} {SURNAME}. В нём указаны ИНН: {INN}, паспорт: '
           '{PASSPORT}, e-mail: {EMAIL}, IP: {IP}. Карта {BANK_CARD}; ОГРНИП {OGRNIP}. '
           'Свидетельство о рождении: {BIRTH_CERT}; КЛАДР: {KLADR}.',
           'Обращение направлено к {NAME} {SURNAME}. Реквизиты: ИНН {INN}, карта '
           '{BANK_CARD}, паспорт {PASSPORT}, ОГРНИП {OGRNIP}. Адрес почты {EMAIL}, IP '
           '{IP}. Свидетельство о рождении {BIRTH_CERT}; код КЛАДР {KLADR}.',
           'Мы говорили о {NAME} {SURNAME}. Паспорт {PASSPORT}; ИНН {INN}; ОГРНИП '
           '{OGRNIP}. Для оплаты используется карта {BANK_CARD}. Контакт: {EMAIL}; '
           'сервер {IP}. Свидетельство о рождении {BIRTH_CERT}. КЛАДР {KLADR}.',
           'Документы подписаны {NAME} {SURNAME}. Паспорт {PASSPORT}, карта {BANK_CARD}, '
           'ИНН {INN}, ОГРНИП {OGRNIP}, e-mail {EMAIL}, IP {IP}, свидетельство о '
           'рождении {BIRTH_CERT}, КЛАДР {KLADR}.'],
 'validation': ['Заявитель {SURNAME} {NAME} уточнил сведения. Паспорт: {PASSPORT}; ИНН: '
                '{INN}; карта: {BANK_CARD}. ОГРНИП: {OGRNIP}; адрес {EMAIL}; IP {IP}; '
                'свидетельство о рождении {BIRTH_CERT}; КЛАДР {KLADR}.',
                'По запросу {NAME} {SURNAME} подготовлена выписка: ИНН {INN}, ОГРНИП '
                '{OGRNIP}, паспорт {PASSPORT}, банковская карта {BANK_CARD}. Почта '
                '{EMAIL}, IP {IP}. Свидетельство о рождении {BIRTH_CERT}; код КЛАДР '
                '{KLADR}.'],
 'test': ['Получатель уведомления — {NAME} {SURNAME}. В карточке клиента записаны '
          'паспорт {PASSPORT} и ИНН {INN}. Платёжная карта: {BANK_CARD}; регистрационный '
          'ОГРНИП: {OGRNIP}. Ответ направить на {EMAIL}. Соединение с {IP}. '
          'Свидетельство о рождении {BIRTH_CERT}; код КЛАДР {KLADR}.',
          'Для {NAME} {SURNAME} сформирован договор. Его реквизиты перечислены ниже:\n'
          'ИНН={INN}; ОГРНИП={OGRNIP}\n'
          'Паспорт {PASSPORT}\n'
          'Карта {BANK_CARD}\n'
          'Email {EMAIL}\n'
          'IP {IP}\n'
          'Свидетельство о рождении {BIRTH_CERT}\n'
          'КЛАДР={KLADR}',
          'Информация о {NAME} {SURNAME} передана оператору. Приложены паспорт '
          '{PASSPORT}, свидетельство о рождении {BIRTH_CERT}, ИНН {INN}, ОГРНИП '
          '{OGRNIP}. Указаны карта {BANK_CARD}, почта {EMAIL}, IP {IP} и код КЛАДР '
          '{KLADR}.',
          'Заявление принято от {NAME} {SURNAME}; обработка займёт два дня. Для сверки: '
          'паспорт {PASSPORT}, ИНН {INN}, ОГРНИП {OGRNIP}. Банковская карта {BANK_CARD}. '
          'Уведомление: {EMAIL}, источник запроса {IP}. Свидетельство о рождении '
          '{BIRTH_CERT}, КЛАДР {KLADR}.',
          'Сотрудник встретился с {NAME} {SURNAME} и проверил карточку. ИНН {INN}; '
          'паспорт {PASSPORT}; ОГРНИП {OGRNIP}; карта {BANK_CARD}; электронный адрес '
          '{EMAIL}; IP-адрес {IP}; свидетельство о рождении {BIRTH_CERT}; КЛАДР {KLADR}.',
          '{SURNAME} {NAME} подтвердил получение заказа. В анкете содержатся ИНН {INN}, '
          'паспорт {PASSPORT}, карта {BANK_CARD}, ОГРНИП {OGRNIP}, e-mail {EMAIL}, IP '
          '{IP}, свидетельство о рождении {BIRTH_CERT} и код КЛАДР {KLADR}.']}

CASES = {'train': ['nomn', 'gent', 'datv', 'loct', 'ablt'],
 'validation': ['nomn', 'gent'],
 'test': ['nomn', 'gent', 'loct', 'gent', 'ablt', 'nomn']}

def synthetic(split, n, seed):
    from faker import Faker
    rng = random.Random(seed)
    fake = Faker('ru_RU')
    fake.seed_instance(seed)
    rows = []

    def number(k):
        return ''.join((rng.choice('0123456789') for _ in range(k)))
    for i in range(n):
        j = i % len(TEMPLATES[split])
        case = CASES[split][j]
        female = i % 2 == 0
        name = fake.first_name_female() if female else fake.first_name_male()
        surname = fake.last_name_female() if female else fake.last_name_male()
        for key, v in (('name', name), ('surname', surname)):
            ps = [p for p in morph().parse(v) if ('Name' if key == 'name' else 'Surn') in p.tag]
            p = (ps or morph().parse(v))[0]
            form = p.inflect({case})
            if key == 'name':
                name = (form.word if form else v).title()
            else:
                surname = (form.word if form else v).title()
        tax = rng.choice(('7701', '7802', '5003', '6604'))
        inn = inn_complete(tax + number(6))
        base = '3' + str(rng.randrange(10, 26)) + rng.choice(('77', '78', '50', '66')) + number(9)
        ogrnip = base + str(int(base) % 13 % 10)
        card = complete_card(rng.choice(('427600', '546900', '220220')) + number(9))
        card = ' '.join((card[k:k + 4] for k in range(0, 16, 4))) if i % 3 else card
        passport = rng.choice(('45', '40', '46', '65')) + str(rng.randrange(10, 26)) + ' ' + number(6)
        if i % 3 == 1:
            passport = passport[:2] + ' ' + passport[2:]
        ip = f'{rng.randrange(10, 220)}.{rng.randrange(256)}.{rng.randrange(256)}.{rng.randrange(1, 255)}' if i % 7 else f'2001:db8:{rng.randrange(65536):x}::{rng.randrange(1, 65536):x}'
        length = (13, 17, 19)[i % 3]
        kladr = rng.choice(('77', '78', '50', '66')) + number(length - 4) + '00'
        values = {'NAME': name, 'SURNAME': surname, 'INN': inn, 'PASSPORT': passport, 'BANK_CARD': card, 'OGRNIP': ogrnip, 'IP': ip, 'EMAIL': f'client{seed}{i}.info@' + rng.choice(('example.org', 'mail.ru', 'example.com')), 'BIRTH_CERT': rng.choice(('IV', 'II', 'VI')) + '-' + rng.choice(('МЮ', 'АГ', 'МУ')) + ' № ' + number(6), 'KLADR': kladr}
        text, spans = render(TEMPLATES[split][j], values)
        if female and 'подтвердил' in text:
            text = text.replace('подтвердил', 'подтвердила')
        template = TEMPLATES[split][j].replace('подтвердил', 'подтвердила') if female else TEMPLATES[split][j]
        text, spans = render(template, values)
        to_bio(text, spans)
        rows.append({'id': f'synthetic_{split}_{i:04d}', 'source': 'synthetic', 'template': f'{split}_{j}', 'text': text, 'entities': spans})
    return rows

def negatives(split, n, seed):
    rng = random.Random(seed)
    rows = []
    templates = ['Вчера завершён проект; номер заявки {v}. Персональные сведения отсутствуют.', 'Мы прочитали интересный роман. Надежда на успех сохраняется. Вера в результат важна.', 'IP: 999.12.34.56. ИНН: {bad}. Номер не прошёл проверку.', 'ОГРНИП {ogrn}. Это испорченная запись.']
    for i in range(n):
        base = '7701' + ''.join((rng.choice('0123456789') for _ in range(6)))
        valid = inn_complete(base)
        bad = valid[:-1] + str((int(valid[-1]) + 1) % 10)
        obase = '32477' + ''.join((rng.choice('0123456789') for _ in range(9)))
        ogrn = obase + str((int(obase) % 13 % 10 + 1) % 10)
        if valid_card(ogrn):
            ogrn = obase + str((int(obase) % 13 % 10 + 2) % 10)
        text = templates[i % len(templates)].format(v=rng.randrange(100, 999999), bad=bad, ogrn=ogrn) + f' Проверка {seed}-{i}.'
        rows.append({'id': f'negative_{split}_{i}', 'source': 'negative', 'text': text, 'entities': []})
    return rows

def prepare_dataset():
    raw = ROOT / 'data/raw'
    raw.mkdir(parents=True, exist_ok=True)
    archive = raw / 'factRuEval-master.zip'
    if not archive.exists():
        urllib.request.urlretrieve('https://codeload.github.com/dialogue-evaluation/factRuEval-2016/zip/refs/heads/master', archive)
    folder = raw / 'factRuEval-2016-master'
    if not folder.exists():
        with zipfile.ZipFile(archive) as z:
            z.extractall(raw)
    dev, dc = fact_documents(folder / 'devset')
    test, tc = fact_documents(folder / 'testset')
    rng = random.Random(20261124)
    rng.shuffle(dev)
    cut = int(len(dev) * 0.8)
    splits = {'train': dev[:cut] + synthetic('train', 800, 12001) + negatives('train', 80, 12002), 'validation': dev[cut:] + synthetic('validation', 120, 12003) + negatives('validation', 24, 12004), 'test': test + synthetic('test', 240, 12005) + negatives('test', 48, 12006)}
    stats = {'seed': 20261124, 'fact_archive_sha256': hashlib.sha256(archive.read_bytes()).hexdigest(), 'annotation': 'Person name/surname spans; other classes inserted with independent offsets', 'overlap_resolution': dc + tc, 'splits': {}}
    for split, rows in splits.items():
        seen = set()
        unique = []
        for row in rows:
            if row['text'] in seen:
                continue
            seen.add(row['text'])
            unique.append(row)
        splits[split] = unique
        write_jsonl(ROOT / f'data/{split}.jsonl', unique)
        counts = Counter((s[0] for row in unique for s in row['entities']))
        docs = Counter((l for row in unique for l in {s[0] for s in row['entities']}))
        stats['splits'][split] = {'documents': len(unique), 'sources': dict(Counter((x['source'] for x in unique))), 'entities': dict(counts), 'documents_per_label': dict(docs), 'sha256': hashlib.sha256((ROOT / f'data/{split}.jsonl').read_bytes()).hexdigest()}
    cross = []
    for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
        overlap = {x['text'] for x in splits[a]} & {x['text'] for x in splits[b]}
        cross.append({'splits': [a, b], 'duplicate_texts': len(overlap)})
        assert not overlap, (a, b)
    stats['cross_split_checks'] = cross
    assert len(splits['test']) >= 100
    assert all((stats['splits']['test']['entities'].get(l, 0) >= 50 for l in LABELS))
    write_json(ROOT / 'data/dataset_stats.json', stats)
    print(stats['splits'])


In [9]:
if not all((DATA_DIR / (split + ".jsonl")).exists() for split in ("train", "validation", "test")):
    prepare_dataset()
dataset_stats = json.loads((DATA_DIR / "dataset_stats.json").read_text(encoding="utf-8"))
for split, values in dataset_stats["splits"].items():
    print(split, values["documents"], values["sources"])


train 977 {'factRuEval2016': 97, 'synthetic': 800, 'negative': 80}
validation 169 {'factRuEval2016': 25, 'synthetic': 120, 'negative': 24}
test 420 {'factRuEval2016': 132, 'synthetic': 240, 'negative': 48}


In [10]:
import argparse

import copy

import hashlib

import json

import random

import sys

import time

from pathlib import Path

BASE = 'DeepPavlov/rubert-base-cased'

TAGS = ['O', 'B-NAME', 'I-NAME', 'B-SURNAME', 'I-SURNAME']

def train_person_model():
    import numpy as np
    import torch
    from torch.utils.data import DataLoader
    from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, get_linear_schedule_with_warmup
    from sklearn.metrics import f1_score
    parser = argparse.ArgumentParser()
    parser.add_argument('--epochs', type=int, default=8)
    parser.add_argument('--batch-size', type=int, default=4)
    parser.add_argument('--lr', type=float, default=5e-05)
    parser.add_argument('--train-layers', type=int, default=4)
    args = parser.parse_args([])
    torch.set_num_threads(4)
    random.seed(12012)
    np.random.seed(12012)
    torch.manual_seed(12012)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(12012)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    root = WORKDIR
    tokenizer = AutoTokenizer.from_pretrained(BASE, use_fast=True)
    model = AutoModelForTokenClassification.from_pretrained(BASE, num_labels=len(TAGS), id2label=dict(enumerate(TAGS)), label2id={x: i for i, x in enumerate(TAGS)})
    for p in model.bert.embeddings.parameters():
        p.requires_grad = False
    for layer in model.bert.encoder.layer[:-args.train_layers]:
        for p in layer.parameters():
            p.requires_grad = False
    model.to(device)

    def encode(rows):
        result = []
        for row in rows:
            spans = [s for s in row['entities'] if s[0] in ('NAME', 'SURNAME')]
            tokens, tags = to_bio(row['text'], spans)
            if not tokens:
                continue
            enc = tokenizer([t[0] for t in tokens], is_split_into_words=True, truncation=True, max_length=256, stride=48, return_overflowing_tokens=True)
            for i in range(len(enc['input_ids'])):
                ids = enc.word_ids(i)
                labels = []
                for k, w in enumerate(ids):
                    labels.append(-100 if w is None or (k > 0 and ids[k - 1] == w) else TAGS.index(tags[w]))
                result.append({**{k: enc[k][i] for k in ('input_ids', 'attention_mask', 'token_type_ids') if k in enc}, 'labels': labels})
        return result
    train = encode(read_jsonl(root / 'data/train.jsonl'))
    val = encode(read_jsonl(root / 'data/validation.jsonl'))
    collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of=8)
    loader = DataLoader(train, batch_size=args.batch_size, shuffle=True, collate_fn=collator, num_workers=0)
    vloader = DataLoader(val, batch_size=8, shuffle=False, collate_fn=collator, num_workers=0)
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=args.lr, weight_decay=0.01)
    total = (len(loader) + 1) // 2 * args.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total * 0.1), total)
    scaler = torch.amp.GradScaler('cuda', enabled=device == 'cuda')
    best = -1.0
    history = []
    start = time.time()
    dest = root / 'models/person'
    dest.mkdir(parents=True, exist_ok=True)
    print(json.dumps({'device': device, 'train_windows': len(train), 'validation_windows': len(val), 'trainable_parameters': sum((p.numel() for p in model.parameters() if p.requires_grad)), 'configuration': vars(args)}), flush=True)
    for epoch in range(args.epochs):
        model.train()
        losses = []
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=device == 'cuda'):
                loss = model(**batch).loss / 2
            scaler.scale(loss).backward()
            losses.append(float(loss.detach()) * 2)
            if step % 2 == 1 or step == len(loader) - 1:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            if (step + 1) % 100 == 0:
                print(f'epoch {epoch + 1} step {step + 1}/{len(loader)} loss {sum(losses[-100:]) / len(losses[-100:]):.4f}', flush=True)
        model.eval()
        gold = []
        pred = []
        with torch.inference_mode():
            for batch in vloader:
                labels = batch.pop('labels')
                out = model(**{k: v.to(device) for k, v in batch.items()}).logits.argmax(-1).cpu()
                mask = labels != -100
                gold.extend(labels[mask].tolist())
                pred.extend(out[mask].tolist())
        wf = f1_score(gold, pred, labels=list(range(1, len(TAGS))), average='weighted', zero_division=0)
        mf = f1_score(gold, pred, labels=sorted(set(gold) - {0}), average='macro', zero_division=0)
        row = {'epoch': epoch + 1, 'train_loss': sum(losses) / len(losses), 'validation_bio_weighted_f1': wf, 'validation_bio_macro_f1': mf, 'elapsed_seconds': time.time() - start}
        history.append(row)
        print(json.dumps(row), flush=True)
        if wf > best:
            best = wf
            model.save_pretrained(dest, safe_serialization=True)
            tokenizer.save_pretrained(dest)
        write_json(root / 'results/training.json', {'base_model': BASE, 'seed': 12012, 'device': device, 'hardware': torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU', 'configuration': vars(args), 'train_windows': len(train), 'validation_windows': len(val), 'trainable_parameters': sum((p.numel() for p in model.parameters() if p.requires_grad)), 'selection': 'maximum validation BIO weighted F1; test not used', 'history': history, 'best_validation_f1': best, 'train_sha256': hashlib.sha256((root / 'data/train.jsonl').read_bytes()).hexdigest(), 'validation_sha256': hashlib.sha256((root / 'data/validation.jsonl').read_bytes()).hexdigest()})
    write_json(dest / 'provenance.json', json.loads((root / 'results/training.json').read_text(encoding='utf-8')))


In [11]:
RUN_TRAINING = False
if RUN_TRAINING:
    train_person_model()


In [ ]:
import hashlib
import shutil
import tempfile
import urllib.request

MODEL_ARCHIVE_URL = 'https://github.com/Champ1y/km2-pii-recognition/releases/download/km2/km2_variant12.zip'
MODEL_SHA256 = '01ac176f1640edeec665138b9ce3e4169ba3caf6ef9ceb41e3387f09759833fd'
weights = MODEL_DIR / "model.safetensors"
if not weights.exists():
    with tempfile.TemporaryDirectory(dir=WORKDIR) as temporary:
        archive_path = Path(temporary) / "km2_variant12.zip"
        print("Загрузка обученной модели")
        with urllib.request.urlopen(MODEL_ARCHIVE_URL, timeout=60) as response, archive_path.open("wb") as output:
            shutil.copyfileobj(response, output, length=1024 * 1024)
        with zipfile.ZipFile(archive_path) as archive:
            for member in archive.infolist():
                model_prefix = "km2_variant12/models/person/"
                if member.is_dir() or not member.filename.startswith(model_prefix):
                    continue
                relative = Path(member.filename.removeprefix(model_prefix))
                if relative.is_absolute() or ".." in relative.parts:
                    raise ValueError("Недопустимый путь в архиве")
                target = MODEL_DIR / relative
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination, length=1024 * 1024)
    if not weights.exists():
        raise FileNotFoundError(str(weights))
    digest = hashlib.sha256()
    with weights.open("rb") as model_file:
        for chunk in iter(lambda: model_file.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != MODEL_SHA256:
        raise ValueError("Контрольная сумма модели не совпадает")
print("Модель:", MODEL_DIR)


In [13]:
import torch
torch.set_num_threads(4)
detector = HybridDetector(MODEL_DIR)
text = 'Иван Иванов подписал заявление. Иван Иванов получил ответ.\nАнна сообщила: «Женя пришла вовремя».\nПаспорт 45 18 123456; ИНН 770698959438.\nБанковская карта 4561 2612 1234 5467; ОГРНИП 304500116000157.\nПочта ivan.petrov@example.org; IP 192.168.1.10.\nСвидетельство о рождении IV-МЮ № 123456; код КЛАДР 77000000000123400.'
entities = detector(text)
for entity in entities:
    print(entity.label, entity.start, entity.end, text[entity.start:entity.end])


C:\Users\artizi\Desktop\kp_tzi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NAME 0 4 Иван
SURNAME 5 11 Иванов
NAME 32 36 Иван
SURNAME 37 43 Иванов
NAME 59 63 Анна
NAME 75 79 Женя
PASSPORT 105 117 45 18 123456
INN 123 135 770698959438
BANK_CARD 154 173 4561 2612 1234 5467
OGRNIP 182 197 304500116000157
EMAIL 205 228 ivan.petrov@example.org
IP 233 245 192.168.1.10
BIRTH_CERT 272 286 IV-МЮ № 123456
KLADR 298 315 77000000000123400


In [14]:
import csv

import hashlib

import json

import sys

import time

from pathlib import Path

def evaluate_dataset():
    import torch
    torch.set_num_threads(4)
    root = WORKDIR
    rows = read_jsonl(root / 'data/test.jsonl')
    detector = HybridDetector(root / 'models/person')
    preds = []
    start = time.time()
    for i, row in enumerate(rows):
        preds.append(detector(row['text']))
        if (i + 1) % 25 == 0:
            print(f'Detected {i + 1}/{len(rows)}, {time.time() - start:.1f}s', flush=True)
    metrics, errors = evaluate(rows, preds)
    metrics['elapsed_seconds'] = time.time() - start
    metrics['test_sha256'] = hashlib.sha256((root / 'data/test.jsonl').read_bytes()).hexdigest()
    metrics['model_sha256'] = hashlib.sha256((root / 'models/person/model.safetensors').read_bytes()).hexdigest()
    write_json(root / 'results/recognition.json', metrics)
    write_jsonl(root / 'results/recognition_errors.jsonl', errors)
    write_jsonl(root / 'results/predictions.jsonl', [{'id': r['id'], 'entities': [s.triple() for s in p]} for r, p in zip(rows, preds)])
    subsets = {}
    for source in ('factRuEval2016', 'synthetic', 'negative'):
        ids = [i for i, r in enumerate(rows) if r['source'] == source]
        subsets[source] = evaluate([rows[i] for i in ids], [preds[i] for i in ids])[0]
    write_json(root / 'results/recognition_subsets.json', subsets)
    baseline, _ = evaluate(rows, [structured(r['text']) for r in rows])
    write_json(root / 'results/rules_only.json', baseline)
    with (root / 'results/recognition_per_attribute.csv').open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.writer(f)
        w.writerow(['attribute', 'TP', 'FP', 'FN', 'support', 'precision', 'recall', 'F1'])
        for l in LABELS:
            m = metrics['per_attribute'][l]
            w.writerow([l] + [m[k] for k in ('tp', 'fp', 'fn', 'support', 'precision', 'recall', 'f1')])
    print(json.dumps({'bio_weighted': metrics['bio']['weighted avg'], 'bio_macro': metrics['bio']['macro avg'], 'per_attribute': metrics['per_attribute'], 'thresholds': metrics['thresholds']}, ensure_ascii=False, indent=2), flush=True)


In [15]:
import gc
del detector
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
evaluate_dataset()
metrics = json.loads((RESULTS_DIR / "recognition.json").read_text(encoding="utf-8"))
print("Weighted BIO F1:", metrics["bio"]["weighted avg"]["f1-score"])
print("Macro BIO F1:", metrics["bio"]["macro avg"]["f1-score"])
for label, values in metrics["per_attribute"].items():
    print(label, "P =", round(values["precision"], 4),
          "R =", round(values["recall"], 4), "F1 =", round(values["f1"], 4))


Detected 25/420, 6.2s


Detected 50/420, 9.2s


Detected 75/420, 11.9s


Detected 100/420, 14.8s


Detected 125/420, 17.6s


Detected 150/420, 19.2s


... (8 outputs left out)

Detected 375/420, 26.6s


Detected 400/420, 27.0s


{
  "bio_weighted": {
    "precision": 0.9844632285144456,
    "recall": 0.984260696076258,
    "f1-score": 0.984290802398001,
    "support": 9022.0
  },
  "bio_macro": {
    "precision": 0.9310287204456058,
    "recall": 0.9302420162740921,
    "f1-score": 0.9306013732654403,
    "support": 9022.0
  },
  "per_attribute": {
    "SURNAME": {
      "tp": 1441,
      "fp": 92,
      "fn": 56,
      "support": 1497,
      "precision": 0.9399869536855838,
      "recall": 0.9625918503674015,
      "f1": 0.9511551155115513
    },
    "NAME": {
      "tp": 961,
      "fp": 41,
      "fn": 82,
      "support": 1043,
      "precision": 0.9590818363273453,
      "recall": 0.9213806327900288,
      "f1": 0.9398533007334964
    },
    "INN": {
      "tp": 240,
      "fp": 0,
      "fn": 0,
      "support": 240,
      "precision": 1.0,
      "recall": 1.0,
      "f1": 1.0
    },
    "PASSPORT": {
      "tp": 560,
      "fp": 0,
      "fn": 0,
      "support": 560,
      "precision": 1.0,
      "reca

Weighted BIO F1: 0.984290802398001
Macro BIO F1: 0.9306013732654403
SURNAME P = 0.94 R = 0.9626 F1 = 0.9512
NAME P = 0.9591 R = 0.9214 F1 = 0.9399
INN P = 1.0 R = 1.0 F1 = 1.0
PASSPORT P = 1.0 R = 1.0 F1 = 1.0
BANK_CARD P = 1.0 R = 1.0 F1 = 1.0
OGRNIP P = 1.0 R = 1.0 F1 = 1.0
IP P = 1.0 R = 1.0 F1 = 1.0
EMAIL P = 1.0 R = 1.0 F1 = 1.0
BIRTH_CERT P = 1.0 R = 1.0 F1 = 1.0
KLADR P = 1.0 R = 1.0 F1 = 1.0
